# 03 - Model Architecture Design: 1D Temporal Convolutional Network

This notebook documents and verifies the denoising model architecture for Phase 3.

**Objective:** Design a denoising model achieving strong HbO-EEG correlation improvement while fitting within MCU memory constraints (~50 KB weights post-quantization).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.tcn import DenoiseTCN, CausalConv1d
from src.models.loss import DenoiseLoss

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Architecture Overview

The model is a 1D Temporal Convolutional Network (TCN) with dilated causal convolutions and residual connections.

```
Input:  [batch, 2, 128]   (HbO + HbR, 128 samples at 10 Hz)
    |
    +-- CausalConv1d(2,  32, k=7, d=1)  + ReLU + residual
    +-- CausalConv1d(32, 32, k=7, d=2)  + ReLU + residual
    +-- CausalConv1d(32, 32, k=7, d=4)  + ReLU + residual
    +-- CausalConv1d(32, 32, k=7, d=8)  + ReLU + residual
    +-- Conv1d(32, 2, k=1)              (projection)
    |
Output: [batch, 2, 128]   (denoised HbO + HbR)
```

**Why TCN over LSTM:**
- No recurrent state: trivial sliding-window inference on MCU
- Fixed receptive field: predictable latency budget
- Highly quantization-friendly (no gating nonlinearities)

## 2. Model Instantiation and Parameter Budget

In [ ]:
model = DenoiseTCN()
print(model)
print(f"\nReceptive field: {model.receptive_field} samples ({model.receptive_field / 10.0:.1f}s at 10 Hz)")
print(f"Total parameters: {model.param_count():,}")
print(f"Size (FP32): {model.size_kb(4):.1f} KB")
print(f"Size (INT8):  {model.size_kb(1):.1f} KB")

In [ ]:
# Per-layer parameter breakdown
print(f"{'Layer':<45} {'Params':>8}  {'KB (FP32)':>10}  {'KB (INT8)':>10}")
print("-" * 80)
total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f"{name:<45} {n:>8,}  {n*4/1024:>10.1f}  {n/1024:>10.2f}")
print("-" * 80)
print(f"{'TOTAL':<45} {total:>8,}  {total*4/1024:>10.1f}  {total/1024:>10.2f}")

**Parameter budget:** With hidden_channels=32, the model has ~22,000 parameters. The INT8 model fits in ~22 KB, well under the 50 KB MCU target. The receptive field of 91 samples (9.1s) covers the hemodynamic response timescale as intended.

## 3. Forward Pass Verification

In [ ]:
# Verify shapes with a dummy batch
x = torch.randn(64, 2, 128)
y = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")
assert y.shape == x.shape, "Shape mismatch!"
print("Shape check passed.")

## 4. Causality Verification

Causal convolutions must not leak future information — changing a future input sample should not affect earlier outputs.

In [ ]:
model.eval()
x1 = torch.randn(1, 2, 128)
x2 = x1.clone()
x2[:, :, 64:] = torch.randn(1, 2, 64)  # perturb future half

with torch.no_grad():
    y1 = model(x1)
    y2 = model(x2)

# Outputs for t < 64 must be identical
diff_past = (y1[:, :, :64] - y2[:, :, :64]).abs().max().item()
diff_future = (y1[:, :, 64:] - y2[:, :, 64:]).abs().max().item()

print(f"Max diff in past outputs (t < 64):   {diff_past:.2e}  (should be ~0)")
print(f"Max diff in future outputs (t >= 64): {diff_future:.2e}  (should be > 0)")
assert diff_past < 1e-6, "Causality violated!"
print("Causality check passed.")

## 5. Impulse Response

Visualize what the (untrained) model does to a single impulse — confirms the receptive field width and causal structure.

In [ ]:
model.eval()
impulse = torch.zeros(1, 2, 128)
impulse[:, 0, 64] = 1.0  # impulse on HbO at t=64

with torch.no_grad():
    response = model(impulse)

time = np.arange(128) / 10.0
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.stem(time, impulse[0, 0].numpy(), linefmt='k-', markerfmt='ko', basefmt='k-', label='Input (HbO)')
ax1.set_title('Input: Impulse at t=6.4s on HbO channel')
ax1.set_ylabel('Amplitude')
ax1.legend()

ax2.plot(time, response[0, 0].numpy(), 'r-', label='HbO response')
ax2.plot(time, response[0, 1].numpy(), 'b-', label='HbR response')
ax2.axvline(x=6.4, color='k', linestyle='--', alpha=0.5, label='Impulse time')
ax2.set_title('Output: Impulse Response (untrained weights)')
ax2.set_ylabel('Amplitude')
ax2.set_xlabel('Time (s)')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Loss Function

```
L = MSE(y_hat, y) + lambda * MSE(|FFT(y_hat)|, |FFT(y)|)
```

The frequency-domain term (lambda=0.1) penalizes spectral distortion, preventing the model from over-smoothing high-frequency HRF components.

In [ ]:
criterion = DenoiseLoss(lambda_freq=0.1)

# Demonstrate with a synthetic example: clean sine + noisy version
t = torch.linspace(0, 12.8, 128).unsqueeze(0).unsqueeze(0).expand(1, 2, -1)
clean = torch.sin(2 * np.pi * 0.08 * t)  # ~0.08 Hz hemodynamic-like
noisy = clean + 0.3 * torch.randn_like(clean)

# Perfect prediction vs noisy prediction
loss_perfect = criterion(clean, clean)
loss_noisy = criterion(noisy, clean)
print(f"Loss (perfect prediction): {loss_perfect.item():.6f}")
print(f"Loss (noisy prediction):   {loss_noisy.item():.4f}")

In [ ]:
# Visualize what the frequency term penalizes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

freqs = np.fft.rfftfreq(128, d=1/10.0)
mag_clean = torch.abs(torch.fft.rfft(clean, dim=-1))[0, 0].numpy()
mag_noisy = torch.abs(torch.fft.rfft(noisy, dim=-1))[0, 0].numpy()

ax1.plot(freqs, mag_clean, 'g-', linewidth=2, label='Clean')
ax1.plot(freqs, mag_noisy, 'r-', alpha=0.7, label='Noisy')
ax1.set_title('Magnitude Spectrum')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('|FFT|')
ax1.legend()

ax2.bar(freqs, (mag_noisy - mag_clean)**2, width=0.03, color='orange', alpha=0.8)
ax2.set_title('Spectral MSE per Frequency Bin')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Squared Error')

plt.tight_layout()
plt.show()

## 7. Gradient Flow Check

Verify that gradients flow through all layers during backpropagation.

In [ ]:
model.train()
model.zero_grad()

x = torch.randn(8, 2, 128)
target = torch.randn(8, 2, 128)
pred = model(x)
loss = criterion(pred, target)
loss.backward()

print(f"{'Layer':<45} {'Grad norm':>12}  {'Grad zero?':>10}")
print("-" * 70)
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        print(f"{name:<45} {grad_norm:>12.6f}  {'YES' if grad_norm == 0 else 'no':>10}")
    else:
        print(f"{name:<45} {'None':>12}  {'N/A':>10}")

## Summary

| Property | Value |
|----------|-------|
| Architecture | 1D TCN with dilated causal convolutions + residual |
| Input/Output | [batch, 2, 128] (HbO + HbR) |
| Hidden channels | 32 |
| Dilations | (1, 2, 4, 8) |
| Kernel size | 7 |
| Receptive field | 91 samples (9.1s at 10 Hz) |
| Parameters | ~22,000 |
| Size (FP32) | ~87 KB |
| Size (INT8) | ~22 KB |
| Loss | MSE + 0.1 * Spectral MSE |

The model is ready for training in Phase 4.